In [397]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

In [398]:
df = pd.read_csv("quote_dataset.csv")

In [399]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [400]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [401]:
df.shape

(3038, 2)

In [402]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


# Apply NLP

## Text Tokenization with Keras

### Steps

1. **Import Tokenizer**: load `Tokenizer` from `tensorflow.keras.preprocessing.text` to convert text into integer sequences.
2. **Set `vocab_size = 10000`**: caps the vocabulary at the 10,000 most frequent words. Rarer words are ignored when converting to sequences.
3. **Create the tokenizer**: `Tokenizer(num_words=vocab_size)` builds the tokenizer with that limit.
4. **Fit on texts**: `fit_on_texts(quotes)` scans all quotes, counts word frequencies, and assigns each word an index.
5. **Inspect `word_index`**: a dictionary of word → integer ID, ranked by frequency (`the` = 1, `you` = 2, ...).
6. **Check vocabulary size**: `len(word_index)` = 8978 unique words, which is below 10000, so no words are cut off.
7. **View top words**: `list(word_index.items())[:10]` shows the most frequent words, mostly stopwords (the, you, to, and).
8. **Convert to sequences**: `texts_to_sequences(quotes)` replaces each word with its integer ID.
9. **Verify output**: print a few quotes next to their sequences to confirm they line up.

### Example

| Quote | Sequence |
|-------|----------|
| "the world as we have created it..." | `[713, 62, 29, 19, 16, 946, 10, ...]` |

### Next Steps

- **Padding**: `pad_sequences(sequence, maxlen=..., padding="post")` to make all sequences the same length.
- **OOV token**: `Tokenizer(num_words=vocab_size, oov_token="<OOV>")` so rare or unseen words get a placeholder instead of being dropped.
- **Model**: feed padded sequences into `Embedding` → LSTM/GRU → Dense.

### Note

- Fix the typo `tokinizer` → `tokenizer` for cleaner code.

In [403]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator).lower()) # removed punctuation and capitals

In [404]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [405]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [406]:
vocab_size = 8978 # changing to max len that is 8978 instead of 10000 below

tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [407]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [408]:
sequence = tokenizer.texts_to_sequences(quotes)

In [409]:
for i in range(3):
  print("Sequence: ", sequence[i])
  print("Sentence: ", quotes[i])
  print()

Sequence:  [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
Sentence:  “the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”

Sequence:  [947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
Sentence:  “it is our choices harry that show what we truly are far more than our abilities”

Sequence:  [1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]
Sentence:  “there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”



In [410]:
from os import sep
X = []
y = []

for seq in sequence:
  for i in range(len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

In [411]:
print(len(X)) # input sizes are diff in each list so we need to apply the padding
print(len(y))

88308
88308


In [412]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [413]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')

In [414]:
X_padded[:3]

array([[  0,   0,   0, ...,   0,   0,   0],
       [  0,   0,   0, ...,   0,   0, 713],
       [  0,   0,   0, ...,   0, 713,  62]], dtype=int32)

In [415]:

y = np.array(y)

In [416]:
X_padded.shape

(88308, 745)

In [417]:
y.shape

(88308,)

In [418]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [419]:
y.shape

(88308,)

In [420]:
y_one_hot.shape

(88308, 8978)

In [421]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM, Dense

In [422]:
embedding_dim = 50
rnn_units = 128

In [423]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

In [424]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [425]:
rnn_model.summary()

Model: "sequential_18"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_18 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_9 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [426]:

lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [427]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [428]:
lstm_model.summary()

Model: "sequential_19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_19 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_9 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [429]:
epochs = 100
batch_size = 128

# history_lstm = lstm_model.fit(
#     X_padded,
#     y_one_hot,
#     epochs=epochs,
#     batch_size=batch_size,
#     validation_split=0.1
# )

In [430]:
epochs = 100
batch_size = 128


# history_rnn = rnn_model.fit(
#     X_padded,
#     y_one_hot,
#     epochs=epochs,
#     batch_size=batch_size,
#     validation_split=0.1
# )

In [431]:
# from tensorflow.keras.models import load_model

# lstm_model = load_model("lstm_model.h5")


# lstm_model.save("lstm_model.h5")

In [432]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word

In [433]:
# index_to_word

In [434]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [435]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]

  # tokenizer.texts_to_sequences(["the world as we"])
  # -> [[713, 62, 29, 19]]     (list containing ONE sequence)

  # tokenizer.texts_to_sequences(["the world as we"])[0]
  # -> [713, 62, 29, 19]       (just the inner list)

  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

## Next-Word Predictor Function

### Code

```python
def predictor(model, tokenizer, text, max_len):
    text = text.lower()

    seq = tokenizer.texts_to_sequences([text])[0]
    seq = pad_sequences([seq], maxlen=max_len, padding='pre')

    pred = model.predict(seq, verbose=0)
    pred_index = np.argmax(pred)
    return index_to_word[pred_index]
```

### Purpose

Predicts the **next word** for a given piece of text. This is the inference step of a next-word prediction model.

### Line by Line

- **`def predictor(model, tokenizer, text, max_len)`**: takes the trained model, the fitted tokenizer, the input text, and `max_len` (the sequence length used during training).
- **`text = text.lower()`**: lowercases the input to match how the training data was prepared.
- **`tokenizer.texts_to_sequences([text])[0]`**: converts words to integer IDs. The text is wrapped in a list because the method expects a list of texts, and `[0]` extracts the single resulting sequence.
- **`pad_sequences([seq], maxlen=max_len, padding='pre')`**: pads with zeros at the **front** so the sequence is exactly `max_len` long. Longer input is truncated from the front, keeping the most recent words.
- **`model.predict(seq, verbose=0)`**: returns a probability for every word in the vocabulary. `verbose=0` hides the progress bar.
- **`np.argmax(pred)`**: picks the index with the highest probability, the most likely next word.
- **`index_to_word[pred_index]`**: converts that index back into an actual word.

### Example Flow

```text
"the world as we"
   -> [713, 62, 29, 19]                (tokenize)
   -> [0, 0, 0, ..., 713, 62, 29, 19]  (pad to max_len)
   -> model outputs probabilities
   -> argmax -> 16
   -> index_to_word[16] -> "have"
```

### Things to Watch

- **`index_to_word` is a global**: it is not defined in the function. Build it with:
```python
  index_to_word = {i: w for w, i in tokenizer.word_index.items()}
```
- **Imports needed**: `numpy as np` and `pad_sequences` from `tensorflow.keras.preprocessing.sequence`.
- **`padding='pre'` must match training**: a different padding style at training time will give poor predictions.
- **Unknown words are dropped**: without an `oov_token`, words outside the vocabulary are silently skipped.
- **Index 0 is reserved for padding**: it has no entry in `word_index`, so `index_to_word[0]` raises a `KeyError` if the model predicts it.
- **Greedy prediction**: `argmax` always returns the single top word. For more varied output, sample from the probabilities using temperature or top-k sampling.

In [436]:
from tensorflow.keras.models import load_model

lstm_model = load_model("lstm_model.h5")

In [437]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 745, 50)        │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │        91,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10000)          │     1,290,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,881,650 (7.18 MB)

 Trainable params: 1,881,648 (7.18 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [444]:
seed_text = "what are you"
next_word = predictor(lstm_model,tokenizer,seed_text,max_len)
print(next_word)

worrying


In [445]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [447]:
seed = "the meaning of life"
generated_text = generate_text(lstm_model,tokenizer,seed,max_len,10)
print(generated_text)

the meaning of life is not a wishgranting factory for beliefyou are to be


In [449]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokenizer, f)

In [450]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)